# Fine-Tuning NLLB-200-Distilled-600M for English ↔ Kinyarwanda Translation

This notebook fine-tunes [`facebook/nllb-200-distilled-600M`](https://huggingface.co/facebook/nllb-200-distilled-600M) — a compact, efficient NLLB variant (600M params) — on the [Digital-Umuganda/kinyarwanda-english-machine-translation](https://huggingface.co/datasets/Digital-Umuganda/kinyarwanda-english-machine-translation) dataset (14K+ parallel sentence pairs).

We use LoRA (via PEFT) for memory-efficient training on Google Colab, then export the merged model to **CTranslate2 INT8** format for fast CPU deployment (~1-3s per sentence vs ~4min with the 1.3B model).

**Why 600M instead of 1.3B?**
- **~3.5x fewer parameters** → fits comfortably in RAM, trains faster
- **CTranslate2 has native, well-tested support** for the 600M distilled architecture (the 1.3B variant had broken CT2 conversions due to `tie_word_embeddings` issues)
- **CPU inference: 1-3 seconds** per sentence (INT8) vs ~4 minutes with the 1.3B model on PyTorch
- For a single language pair (EN↔KIN), the 1.3B model is overkill — the 600M distilled model is designed for this

**Open-source projects used:**
- `facebook/nllb-200-distilled-600M` — Base translation model (200 languages incl. Kinyarwanda)
- `Digital-Umuganda/kinyarwanda-english-machine-translation` — 14K+ parallel sentence pairs
- `OpenNMT/CTranslate2` — MIT-licensed fast inference engine for CPU deployment

**Output:** `nllb-kin-ct2/` directory (~300MB) with INT8 model + tokenizer files

In [1]:
# Cell 2: Install dependencies
!pip install -q transformers datasets peft accelerate sentencepiece sacrebleu ctranslate2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 58.1 MB/s eta 0:00:00


In [ ]:
# Cell 3: Load base model and tokenizer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# facebook/nllb-200-distilled-600M — compact NLLB variant (600M params)
# Designed for efficient single-pair translation with CTranslate2 support
MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

In [14]:
# Cell 4: Load dataset, split 90/10 train/test
#
# The DigitalUmuganda TSV is encoded in Windows-1252 (not UTF-8),
# so we download and parse it manually instead of using load_dataset().

import pandas as pd
from datasets import Dataset

TSV_URL = (
    "https://huggingface.co/datasets/DigitalUmuganda/"
    "kinyarwanda-english-machine-translation-dataset/"
    "resolve/main/kinyarwanda-english-corpus.tsv"
)

# Read with cp1252 to handle curly-apostrophe bytes (0x92, 0x93, etc.)
df = pd.read_csv(
    TSV_URL,
    sep="\t",
    header=None,
    names=["rw", "en"],
    encoding="cp1252",
    on_bad_lines="skip",   # drop the handful of malformed rows
)

# Drop rows where either side is empty
df = df.dropna(subset=["rw", "en"])
df["rw"] = df["rw"].str.strip()
df["en"] = df["en"].str.strip()
df = df[(df["rw"].str.len() > 0) & (df["en"].str.len() > 0)]

print(f"Total sentence pairs after cleaning: {len(df):,}")
print(f"\nSample rows:")
print(df.head(3).to_string())

# Convert to HuggingFace Dataset and split 90/10
full_dataset = Dataset.from_pandas(df, preserve_index=False)
split = full_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
test_dataset = split["test"]

print(f"\nTrain samples: {len(train_dataset):,}")
print(f"Test samples:  {len(test_dataset):,}")

Total sentence pairs after cleaning: 14,401

Sample rows:
                              rw                                             en
0  Uruvunge rw’abatuye uwo mujyi  The crowd of the people who live in that city
1      amuhamagara kuri telefone                     he called him on the phone
2    Pawulo yakundaga Timoteyo “                           Paul loved Timothy “

Train samples: 12,960
Test samples:  1,441


In [15]:
# Cell 5: Preprocessing — tokenize source (English) and target (Kinyarwanda) with NLLB lang codes
MAX_LENGTH = 128
SRC_LANG = "eng_Latn"  # English
TGT_LANG = "kin_Latn"  # Kinyarwanda

def preprocess_function(examples):
    """Tokenize English->Kinyarwanda pairs for NLLB."""
    tokenizer.src_lang = SRC_LANG
    inputs = examples["en"]
    targets = examples["rw"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

    # Tokenize targets with Kinyarwanda lang code
    tokenizer.src_lang = TGT_LANG
    labels = tokenizer(
        targets,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

print(f"Tokenized train: {tokenized_train}")
print(f"Tokenized test: {tokenized_test}")

Map:   0%|          | 0/12960 [00:00<?, ? examples/s]

Map:   0%|          | 0/1441 [00:00<?, ? examples/s]

Tokenized train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 12960
})
Tokenized test: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1441
})


In [16]:
# Cell 6: Apply LoRA via PEFT for memory-efficient training
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,718,592 || all params: 2,162,421,760 || trainable%: 0.2182


In [ ]:
# Cell 7: Configure Seq2SeqTrainer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-kin-lora",
    num_train_epochs=3,
    per_device_train_batch_size=16,   # 600M model allows larger batches
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Trainer configured. Ready to train.")

In [22]:
# Cell 8: Train
trainer.train()

Step,Training Loss,Validation Loss
500,9.581719,8.988597
1000,7.686972,7.378880
1500,7.361178,7.125470
2000,7.247375,7.067151
2500,7.218502,7.042251
3000,7.182648,7.030592
3500,7.170302,7.022785
4000,7.158731,7.019746
4500,7.157291,7.016412


TrainOutput(global_step=4860, training_loss=7.834377976405768, metrics={'train_runtime': 1709.7346, 'train_samples_per_second': 22.74, 'train_steps_per_second': 2.843, 'total_flos': 4.106793781297152e+16, 'train_loss': 7.834377976405768, 'epoch': 3.0})

In [23]:
# Cell 9: Evaluate on test set — compute BLEU score with sacrebleu
import numpy as np
import sacrebleu

predictions = trainer.predict(tokenized_test)
pred_ids = predictions.predictions
# Replace -100 with pad token id
pred_ids = np.where(pred_ids != -100, pred_ids, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(
    np.where(predictions.label_ids != -100, predictions.label_ids, tokenizer.pad_token_id),
    skip_special_tokens=True,
)

# Strip whitespace
decoded_preds = [pred.strip() for pred in decoded_preds]
decoded_labels = [label.strip() for label in decoded_labels]

bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
print(f"BLEU score: {bleu.score:.2f}")
print(f"\nSample predictions:")
for i in range(min(5, len(decoded_preds))):
    print(f"  Pred: {decoded_preds[i]}")
    print(f"  Ref:  {decoded_labels[i]}")
    print()

BLEU score: 54.29

Sample predictions:
  Pred: Hariya hari ahantu hizewe
  Ref:  Aha ni ahantu hizewe

  Pred: Si ubwa mbere bakundanye.
  Ref:  Si ubwa mbere bakundanye nubushize bari bar kumwe.

  Pred: Ugomba gupakira ifunguro rya sasita kugirango ugire icyo kurya murugendo rwawe
  Ref:  Ugomba gupakira ifunguro rya sasita kugirango ugire icyo urya mukugenda kwawe

  Pred: Siye ikawa yawe aho kugira ngo ikonje.
  Ref:  Siga ikawa yawe aho kugirango ikonje.

  Pred: Nagize ubwoba ko bitazagenda neza kuko nari nzi neza ko byananiranye.
  Ref:  Natinyaga ko bitagenda neza kubera konzi neza ko byananiranye.



In [35]:
# Cell 10: Test with sample sentences (English→Kinyarwanda and Kinyarwanda→English)

def translate(text, src_lang, tgt_lang):
    """Translate text between English and Kinyarwanda."""
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    translated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=MAX_LENGTH,
        num_beams=4,
    )
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# English → Kinyarwanda
en_samples = [
    "So, you want to know about this matrix thing. Dont worry, Im here to help! A matrix is like an arrangement of numbers or symbols into a square grid. Its called a square array because it has the same number of rows and columns. Think of it like a piece of paper with 3 rows and 3 columns. Each space on that paper can hold a single value, like a number or a symbol. When you fill in those spaces, you get a matrix! In this case, were talking about a special kind of matrix called a "tridiagonal" or "Toeplitz" matrix. But dont worry too much about those names right now. Just know that its a specific way of arranging numbers and symbols into a square grid. The example you provided shows what a 3x3 matrix might look like: ``` a b c d e f g h i ``` Each row has three elements, and each column also has three elements. Its like a little table with numbers or symbols in it! So, to summarize: a matrix is just an arrangement of values into a square grid. The specific example you provided is a 3x3 matrix, which means it has three rows and three columns.",
    "Photosynthesis is the process by which plants make food from sunlight.",
    "The exam will be held next Monday at 9 AM.",
    "Please submit your homework before Friday.",
]

print("=== English → Kinyarwanda ===")
for sentence in en_samples:
    result = translate(sentence, "eng_Latn", "kin_Latn")
    print(f"  EN: {sentence}")
    print(f"  KIN: {result}")
    print()

# Kinyarwanda → English
kin_samples = [
    "Muraho mwarimu, amakuru?",
    "Ishuri ritangira saa mbiri z'igitondo.",
    "Ndashaka kwiga iby'ikoranabuhanga.",
]

print("=== Kinyarwanda → English ===")
for sentence in kin_samples:
    result = translate(sentence, "kin_Latn", "eng_Latn")
    print(f"  KIN: {sentence}")
    print(f"  EN: {result}")
    print()

=== English → Kinyarwanda ===
  EN: So, you want to know about this matrix thing. Dont worry, Im here to help! A matrix is like an arrangement of numbers or symbols into a square grid. Its called a square array because it has the same number of rows and columns. Think of it like a piece of paper with 3 rows and 3 columns. Each space on that paper can hold a single value, like a number or a symbol. When you fill in those spaces, you get a matrix! In this case, were talking about a special kind of matrix called a "tridiagonal" or "Toeplitz" matrix. But dont worry too much about those names right now. Just know that its a specific way of arranging numbers and symbols into a square grid. The example you provided shows what a 3x3 matrix might look like: ``` a b c d e f g h i ``` Each row has three elements, and each column also has three elements. Its like a little table with numbers or symbols in it! So, to summarize: a matrix is just an arrangement of values into a square grid. The specif

In [ ]:
# Cell 11: Merge LoRA weights back into base model, save full model
import json

merged_model = model.merge_and_unload()

MERGED_DIR = "./nllb-kin-merged"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# Fix tie_word_embeddings in config — this caused the 1.3B CT2 conversion to produce garbage.
# The 600M model should be clean, but we set it explicitly to be safe.
config_path = f"{MERGED_DIR}/config.json"
with open(config_path, "r") as f:
    config = json.load(f)
config["tie_word_embeddings"] = True
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Merged model saved to {MERGED_DIR}")
print(f"tie_word_embeddings set to: {config['tie_word_embeddings']}")

In [ ]:
# Cell 12: Convert to CTranslate2 INT8 format for CPU deployment
import ctranslate2
import shutil, glob

CT2_DIR = "./nllb-kin-ct2"

converter = ctranslate2.converters.TransformersConverter(
    model_name_or_path=MERGED_DIR,
)
converter.convert(
    output_dir=CT2_DIR,
    quantization="int8",
    force=True,
)

# Copy tokenizer files needed for the deployment service
for f in glob.glob(f"{MERGED_DIR}/sentencepiece*") + glob.glob(f"{MERGED_DIR}/tokenizer*") + glob.glob(f"{MERGED_DIR}/special_tokens*"):
    shutil.copy2(f, CT2_DIR)

print(f"CTranslate2 INT8 model saved to {CT2_DIR}")
!ls -lh {CT2_DIR}

In [ ]:
# Cell 13: Test CTranslate2 model to verify it works
import ctranslate2
import time

translator = ctranslate2.Translator(CT2_DIR, device="cpu", inter_threads=4)

def ct2_translate(text, src_lang, tgt_lang):
    """Translate using CTranslate2 INT8 model."""
    tokenizer.src_lang = src_lang

    # Tokenize to subword strings
    tokens = tokenizer.tokenize(text)

    # NLLB CTranslate2 format: [src_lang] + tokens + [</s>]
    source = [src_lang] + tokens + ["</s>"]

    results = translator.translate_batch(
        [source],
        target_prefix=[[tgt_lang]],
        beam_size=4,
        max_decoding_length=256,
    )

    # Skip the target language prefix token
    output_tokens = results[0].hypotheses[0][1:]

    # Decode back to text
    token_ids = tokenizer.convert_tokens_to_ids(output_tokens)
    return tokenizer.decode(token_ids, skip_special_tokens=True)

# Test EN → KIN with timing
test_sentences = [
    "Hello teacher, how are you?",
    "The exam is tomorrow morning.",
    "Please open your textbooks to page 42.",
    "Photosynthesis is the process by which plants make food from sunlight.",
]

print("=== CTranslate2 EN → KIN ===")
for sentence in test_sentences:
    start = time.time()
    result = ct2_translate(sentence, "eng_Latn", "kin_Latn")
    elapsed = time.time() - start
    print(f"  EN:  {sentence}")
    print(f"  KIN: {result}")
    print(f"  Time: {elapsed:.2f}s")
    print()

# Test KIN → EN
print("=== CTranslate2 KIN → EN ===")
kin_samples = [
    "Muraho mwarimu, amakuru?",
    "Ishuri ritangira saa mbiri z'igitondo.",
    "Ndashaka kwiga iby'ikoranabuhanga.",
]
for sentence in kin_samples:
    start = time.time()
    result = ct2_translate(sentence, "kin_Latn", "eng_Latn")
    elapsed = time.time() - start
    print(f"  KIN: {sentence}")
    print(f"  EN:  {result}")
    print(f"  Time: {elapsed:.2f}s")
    print()

In [ ]:
# Cell 14: Zip the CTranslate2 model for deployment and download
import shutil

shutil.make_archive("nllb-kin-ct2", "zip", ".", "./nllb-kin-ct2")
print("CTranslate2 model zipped to nllb-kin-ct2.zip")

# Also zip the LoRA checkpoint in case you need it later
shutil.make_archive("nllb-kin-lora", "zip", ".", "./nllb-kin-lora")
print("LoRA checkpoint zipped to nllb-kin-lora.zip")

# Download in Google Colab
try:
    from google.colab import files
    files.download("nllb-kin-ct2.zip")
    files.download("nllb-kin-lora.zip")
except ImportError:
    print("Not running in Colab. Find the zips in the current directory.")